# EVA Memory — Notebook test

1. **Run the first code cell** (path fix) so `src` imports work. It adds the backend root to `sys.path`.
2. Data (SQLite + FAISS) is created under `backend/data/` (or `{backend_root}/data/`).
3. Set **`OPENAI_API_KEY`** in the environment if using OpenAI for embedding/LLM.

In [5]:
# Fix imports from notebook: add backend root (directory containing "src") to sys.path
import sys
from pathlib import Path

def _find_backend_root():
    for candidate in [Path.cwd()] + list(Path.cwd().parents):
        if (candidate / "src" / "eva_memory").is_dir():
            return candidate
    return Path.cwd()

_backend_root = _find_backend_root()
if str(_backend_root) not in sys.path:
    sys.path.insert(0, str(_backend_root))
print("backend root:", _backend_root)

backend root: /Users/athvaithk/Desktop/Athvaith/GEN_AI/Project-Eva/backend


In [6]:
from pathlib import Path
from src.eva_memory import MemoryClient, EvaMemoryConfig, EmbeddingConfig
from src.eva_memory.config import LLMConfig

# Use backend root so paths work regardless of notebook cwd
_data_dir = _backend_root / "data"
config = EvaMemoryConfig(
    sqlite_path=_data_dir / "memories.db",
    faiss_dir=_data_dir / "faiss",
    embedding=EmbeddingConfig(
        model="openai:text-embedding-3-small",
        # base_url only for Ollama, e.g. base_url="http://localhost:11434",
    ),
    llm=LLMConfig(
        model="openai:gpt-4.1-mini",
        temperature=0.2,
        max_tokens=2048,
    ),
)
client = MemoryClient(config=config)

In [7]:
record = client.add(
    user_id="user_123",
    text="User prefers dark mode and large font size.",
    metadata={"category": "assistant_preferences", "importance": "high"},
)
print(record.memory_id, record.text, record.created_at)

b51ce1ae-abc6-4d9f-9d54-1065fa7fa833 User prefers dark mode and large font size. 2026-03-08T08:24:41.708220+00:00


In [8]:
hits = client.search(
    user_id="user_123",
    query="What are their UI preferences?",
    k=5,
    categories=["assistant_preferences"],
)
for h in hits:
    print(h.score, h.memory.text)
records = client.list(user_id="user_123", limit=100)
print("list count:", len(records))

1.1100199222564697 User prefers dark mode and large font size.
list count: 1


In [12]:
context = client.get_context(
    user_id="user_123",
    query="user's current message or a short summary",
    k=5,
    mode="semantic",
    categories=None,
)
print(context if context else "(empty)")

Relevant user context:
- User prefers dark mode and large font size.
- The user follows a vegetarian diet.


In [11]:
import asyncio
from src.eva_memory import Message

messages = [
    Message(role="user", content="I've switched to a vegetarian diet."),
    Message(role="assistant", content="I'll remember that. Any allergies I should know about?"),
]
changes = await client.add_from_messages(
    user_id="user_123",
    messages=messages,
    category_hint="health_wellness",
)
for c in changes:
    print(c.event, c.memory_id, getattr(c, "new_text", None) or getattr(c, "old_text", None))

ADD 47281724-ca0c-485f-bb70-cbb6889ae70d The user follows a vegetarian diet.


In [13]:
import asyncio
from src.eva_memory import Message

messages = [
    Message(role="user", content="I've switched to a non-vegetarian diet."),
    Message(role="assistant", content="I'll remember that. Any allergies I should know about?"),
]
changes = await client.add_from_messages(
    user_id="user_123",
    messages=messages,
    category_hint="health_wellness",
)
for c in changes:
    print(c.event, c.memory_id, getattr(c, "new_text", None) or getattr(c, "old_text", None))

UPDATE 47281724-ca0c-485f-bb70-cbb6889ae70d The user follows a non-vegetarian diet.


In [14]:
context = client.get_context(
    user_id="user_123",
    query="user's current message or a short summary",
    k=5,
    mode="semantic",
    categories=None,
)
print(context if context else "(empty)")

Relevant user context:
- User prefers dark mode and large font size.
- The user follows a non-vegetarian diet.
